In [0]:
%run ./Copy-Datasets

In [0]:
# %run ../Copy-Datasets

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
dataset_bookstore = f"/Volumes/{current_catalog}/bookstore_eng_pro/dataset"
checkpoint_path = f"/Volumes/{current_catalog}/bookstore_eng_pro/checkpoints"
print(dataset_bookstore)

In [0]:
df_books = spark.table("books_silver")
display(df_books)

In [0]:
def apply_discount(price, percentage):
    return price * (1 - percentage/100)

In [0]:
apply_discount(100, 20)

In [0]:
apply_discount_udf = udf(apply_discount)

In [0]:
from pyspark.sql.functions import col, lit

df_discounts = df_books.select("price", apply_discount_udf(col("price"), lit(50)))
display(df_discounts)

In [0]:
apply_discount_py_udf = spark.udf.register("apply_discount_sql_udf", apply_discount)

In [0]:
df_discounts = df_books.select("price", apply_discount_py_udf(col("price"), lit(50)))
display(df_discounts)

In [0]:
%sql
SELECT price, apply_discount_sql_udf(price, 50) AS price_after_discount
FROM books_silver

In [0]:
@udf("double")
def apply_discount_decorator_udf(price, percentage):
    return price * (1 - percentage/100)

In [0]:
df_discounts = df_books.select("price", apply_discount_decorator_udf(col("price"), lit(50)))
display(df_discounts)

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

def vectorized_udf(price: pd.Series, percentage: pd.Series,) -> pd.Series:
    return price * (1 - percentage/100)

vectorized_udf = pandas_udf(vectorized_udf, "double")

In [0]:
@pandas_udf("double")
def vectorized_udf(price: pd.Series, percentage: pd.Series,) -> pd.Series:
    return price * (1 - percentage/100)

In [0]:
df_domains = df_books.select("price", vectorized_udf(col("price"), lit(50)))
display(df_domains)

In [0]:
spark.udf.register("sql_vectorized_udf", vectorized_udf)

In [0]:
%sql
SELECT price, sql_vectorized_udf(price, 50) AS price_after_discount
FROM books_silver